# Zonas de vida de Holdridge (38 zonas) — CHELSA V2.1, normal 1991-2020

Versão em Python (API do Earth Engine) do script JS anterior, adaptada aos **três assets separados por
variável** (`tas`, `pet`, `pr`; 12 bandas mensais cada), gerados por `gerar_normal_multibanda.py`.

**O que mudou em relação ao script antigo:** os assets já estão em unidades físicas
(tas em °C; pet e pr em mm/mês). Por isso **não há mais** `subtract(273.15)`, `divide(100)` nem `divide(10)`.
A biotemperatura, a classificação por classes e a tabela das 38 zonas seguem a lógica do script original.
O cálculo está em [holdridge_gee.py](holdridge_gee.py).

A suavização por moda (focalMode) do script JS foi trazida de volta (seção 3b): `brasil` (contorno real do
país, FAO GAUL) é usado para `clip`/geometria/mapa e também para recortar `zona_final` de volta ao contorno
do país depois da suavização (que pode espalhar valores um pouco além da borda). `zona_final` tem as mesmas
características do que seria exportado manualmente no GEE. Este notebook **não exporta** nada
automaticamente; a exportação fica a cargo do usuário.

## 1. Configuração

In [ ]:
import sys
sys.path.insert(0, ".")

import ee
import geemap
import pandas as pd
import holdridge_gee as h

PROJETO = "fcoliveira"

# Assets separados por variavel (12 bandas mensais cada; ajuste para o caminho onde voce subiu as imagens)
ASSET_TAS = "projects/fcoliveira/assets/chelsa_brasil_tas_normal_1991_2020"
ASSET_PET = "projects/fcoliveira/assets/chelsa_brasil_pet_normal_1991_2020"
ASSET_PR = "projects/fcoliveira/assets/chelsa_brasil_pr_normal_1991_2020"

# Asset de saida com a classificacao (referencia; a exportacao e manual, ver secao 6)
ASSET_SAIDA = "projects/fcoliveira/assets/CHELSA/Holdridge_CHELSA_BR_1991_2020"

# None = correcao de latitude em todos os meses (igual ao script JS original).
# 24  = correcao so nos meses com t > 24 C. Veja a secao 4 antes de decidir.
LIMIAR_CORRECAO = None


## 2. Earth Engine e região (Brasil)

In [ ]:
try:
    ee.Initialize(project=PROJETO)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJETO)

# 'brasil' usa o contorno real do pais (FAO GAUL) para clip/geometria/mapa.
brasil = (ee.FeatureCollection("FAO/GAUL/2015/level0")
          .filter(ee.Filter.eq("ADM0_NAME", "Brazil")))
estados = (ee.FeatureCollection("FAO/GAUL/2015/level1")
           .filter(ee.Filter.eq("ADM0_NAME", "Brazil")))


## 3. Carregar a normal e classificar

In [ ]:
normal = h.carregar_normal(ASSET_TAS, ASSET_PET, ASSET_PR)
print("Bandas:", normal.bandNames().getInfo())

resultado = h.classificar_holdridge(normal, LIMIAR_CORRECAO).clip(brasil)
zona = resultado.select("zone38_id")

## 3b. Suavização (focalMode) e recorte final

Reproduz as etapas finais do script JS original: filtro de moda 3x3 na zona classificada e recorte de
volta ao contorno de `brasil` (a suavização pode espalhar valores por 1 pixel além da borda; o `clip`
final remove essa sobra e também define como "sem zona" (0) os pixels dentro do país que não receberam
classificação). O resultado, `zona_final`, é a imagem pronta para a exportação manual (seção 6).

In [ ]:
zona_final = (h.suavizar_zona(zona).unmask(0).clip(brasil)
              .rename("zone38_id")
              .toByte())


## 4. Verificação: áreas por zona

No Brasil não se espera gelo/polar (zonas 1 e 2). **Se aparecerem em grande área no Sul, é a correção de latitude
aplicada em todos os meses**: com temperaturas em °C corretas, `t - 0,03 * lat * (t - 24)^2` derruba a temperatura
de meses mais frios a zero em latitudes maiores que ~25°. Nesse caso, defina `LIMIAR_CORRECAO = 24` na configuração
e rode de novo.

In [5]:
hist = zona_final.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(),
    geometry=brasil.geometry(),
    scale=5000,
    maxPixels=1e13,
    bestEffort=True,
).get("zone38_id").getInfo()

tabela = (pd.Series(hist, name="pixels_5km").rename_axis("zona").reset_index()
          .assign(zona=lambda d: d.zona.astype(int))
          .sort_values("zona"))
tabela["area_km2"] = tabela["pixels_5km"] * 25
tabela["pct"] = (100 * tabela["pixels_5km"] / tabela["pixels_5km"].sum()).round(2)
display(tabela)

if tabela["zona"].isin([1, 2]).any():
    pct = tabela.loc[tabela["zona"].isin([1, 2]), "pct"].sum()
    print(f"ATENCAO: {pct:.1f}% do Brasil caiu nas zonas 1-2 (gelo/polar). Reveja LIMIAR_CORRECAO.")


,zona,pixels_5km,area_km2,pct
0,0,8.850980,2.212745e+02,0.00
1,1,1431.000000,3.577500e+04,0.41
8,2,144.000000,3.600000e+03,0.04
26,4,1281.000000,3.202500e+04,0.36
27,8,34.086275,8.521569e+02,0.01
28,9,3312.000000,8.280000e+04,0.94
2,10,1223.000000,3.057500e+04,0.35
3,13,2093.121569,5.232804e+04,0.60
4,14,14160.337255,3.540084e+05,4.03
5,15,2845.603922,7.114010e+04,0.81


ATENCAO: 0.4% do Brasil caiu nas zonas 1-2 (gelo/polar). Reveja LIMIAR_CORRECAO.


## 5. Mapa

In [ ]:
vis = {"min": 1, "max": 38, "palette": h.PALETA}

Map = geemap.Map(center=[-14, -52], zoom=4)
Map.addLayer(zona, vis, "Holdridge (38 zonas)")
Map.addLayer(zona_final, vis, "Holdridge (38 zonas kernel)")
Map.addLayer(resultado.select("biotemp"), {"min": 10, "max": 30, "palette": ["blue", "yellow", "red"]}, "Biotemperatura", False)
Map.addLayer(resultado.select("prec"), {"min": 0, "max": 3500, "palette": ["white", "blue"]}, "Precipitacao anual", False)
Map.addLayer(resultado.select("retp"), {"min": 0, "max": 4, "palette": ["green", "yellow", "red"]}, "Razao ETP/P", False)
Map.addLayer(estados.style(color="000000", fillColor="00000000", width=1), {}, "Estados")
Map.addLayer(brasil.style(color="000000", fillColor="00000000", width=2), {}, "Brasil")
Map.add_colorbar(vis, label="Zona de Holdridge (id 1-38)", layer_name="Holdridge (38 zonas)")
Map


## 6. Exportação (manual)

Este notebook **não exporta** nada — a exportação é feita manualmente no GEE depois. A imagem `zona_final`
gerada na seção 3b já tem as mesmas características do que seria exportado manualmente no GEE: recorte
pelo contorno do Brasil (FAO GAUL), suavização por moda (3x3), banda `zone38_id` em `Byte`. Ao subir
manualmente, use `pyramidingPolicy: 'mode'` para essa banda.